# PCI‑MARL Causal Validation Notebook

이 노트북은 custom StagHunt 환경에서 **ground‑truth intervention reward**를 Monte‑Carlo로 계산하기 위한 코드이다.

동일한 hidden state에서 agent i의 action을 강제로 a로 고정하고,
다른 agent는 현재 policy로 행동하도록 하여

E[R_t(a)]

를 직접 추정한다.

## Imports

In [200]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import copy
import numpy as np
import torch
from envs.multi_agent_env import StagHuntEnv

## Intervention Step Function

In [201]:
def step_with_intervention(env, forced_agent, forced_action, policy_fn):
    obs = env._get_observations()
    actions = {}
    for agent_id, o in enumerate(obs):
        if agent_id == forced_agent:
            actions[agent_id]=forced_action
        else:
            actions[agent_id] = policy_fn(agent_id, o)

    next_obs, rewards, done, info = env.step(actions)
    return rewards, next_obs, done, info

## Monte Carlo Intervention Estimator

In [202]:
def estimate_intervention_reward(env, forced_agent, forced_action, policy_fn, mc_samples=100):
    rewards = []
    for _ in range(mc_samples):
        env_clone = copy.deepcopy(env)
        r, _, _, _ = step_with_intervention(env_clone, forced_agent, forced_action, policy_fn)
        rewards.append(sum(r.values()))
    return np.mean(rewards)

## Example Random Policy

In [203]:
def random_policy(agent_id, obs):
    return np.random.randint(5)

## Full Validation Loop

In [204]:
def run_validation(env, agent_id=0, action_space=5, mc_samples=1000):
    results = []
    for action in range(action_space):
        mu = estimate_intervention_reward(env, agent_id, action, random_policy, mc_samples)
        results.append((action, mu))
    return results

## Usage Example

In [205]:
env = StagHuntEnv(num_agents = 3)
env.reset()
results = run_validation(env)
print(results)

[(0, 1.185), (1, 1.173), (2, 1.173), (3, 1.161), (4, 1.182)]
